# VPO Full Process Demo

This notebook walks through the complete runnable scaffold in this repo:

1. Generate synthetic traces from latent dominance-induced partial orders.
2. Inspect frontier-restricted HPO trace probabilities.
3. Train IWVI (with VIMCO-style control variates) to recover the latent order.
4. Evaluate relation recovery quality.
5. Demonstrate PO-DSL order factoring with maximal paths and a scaling stress table.


In [1]:
from pathlib import Path
import sys
import numpy as np

# Allow running from either repo root or notebooks/.
root = Path.cwd().resolve()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
sys.path.insert(0, str(root / "src"))

from vpo.data import make_synthetic_trace_data, relation_f1
from vpo.hpo import HPOModel
from vpo.poset import dominance_relation, frontier, is_strict_partial_order, relation_edges
from vpo.vi import IWVIConfig, IWVITrainer, MeanFieldGaussian
from vpo.order import closure_edges, transitive_reduction
from vpo.po_dsl import build_maximal_paths_from_order, h_prime, path_penalty_naive, order_stats

np.set_printoptions(precision=3, suppress=True)
SEED = 7
rng = np.random.default_rng(SEED)
print(f"Repo root: {root}")


Repo root: /Users/dongqing/Desktop/vi_inference


## 1) Create synthetic traces and latent relation

In [2]:
num_actions = 8
latent_dim = 3
num_traces = 300
beta = 4.0
epsilon = 0.05

data = make_synthetic_trace_data(
    num_actions=num_actions,
    latent_dim=latent_dim,
    num_traces=num_traces,
    beta=beta,
    epsilon=epsilon,
    structured_order=True,
    seed=SEED,
)

true_edges = relation_edges(data.relation)
print(f"Embeddings shape: {data.embeddings.shape}")
print(f"Number of traces: {len(data.traces)}")
print(f"Number of true relation edges: {len(true_edges)}")
print(f"Strict partial order: {is_strict_partial_order(data.relation)}")
print("First 3 traces:")
for t in data.traces[:3]:
    print("  ", t)


Embeddings shape: (8, 3)
Number of traces: 300
Number of true relation edges: 28
Strict partial order: True
First 3 traces:
   [1, 0, 2, 3, 6, 4, 5, 7]
   [0, 1, 2, 3, 4, 5, 6, 7]
   [0, 1, 2, 3, 4, 5, 6, 7]


## 2) Inspect frontier-restricted HPO step probabilities

In [3]:
model = HPOModel(beta=beta, epsilon=epsilon)
trace = data.traces[0]
shuffled = trace.copy()
rng.shuffle(shuffled)

ll_trace = model.trace_log_likelihood(trace, data.relation)
ll_shuffled = model.trace_log_likelihood(shuffled, data.relation)

print("Reference trace:", trace)
print("Shuffled trace :", shuffled)
print(f"log p(reference trace) = {ll_trace:.3f}")
print(f"log p(shuffled trace)  = {ll_shuffled:.3f}")

prefix = []
print("\nFirst few per-step probabilities on the reference trace:")
for action in trace[:5]:
    remaining = [a for a in range(num_actions) if a not in prefix]
    front = frontier(data.relation, remaining)
    p = model.step_probability(action, prefix, data.relation)
    print(f"  prefix={prefix}, frontier={front}, action={action}, p={p:.4f}")
    prefix.append(action)


Reference trace: [1, 0, 2, 3, 6, 4, 5, 7]
Shuffled trace : [1, 5, 7, 2, 6, 4, 0, 3]
log p(reference trace) = -9.644
log p(shuffled trace)  = -27.911

First few per-step probabilities on the reference trace:
  prefix=[], frontier=[0], action=1, p=0.0063
  prefix=[1], frontier=[0], action=0, p=0.9571
  prefix=[1, 0], frontier=[2], action=2, p=0.9583
  prefix=[1, 0, 2], frontier=[3], action=3, p=0.9600
  prefix=[1, 0, 2, 3], frontier=[4], action=6, p=0.0125


## 3) Train IWVI to recover latent order from traces

In [4]:
q = MeanFieldGaussian.initialize(
    num_actions=num_actions,
    latent_dim=latent_dim,
    init_scale=0.1,
    seed=SEED + 1,
)

cfg = IWVIConfig(
    num_particles=8,
    alpha=0.0,
    lr=0.02,
    beta=beta,
    epsilon=epsilon,
)

trainer = IWVITrainer(traces=data.traces, config=cfg)
history = trainer.fit(q, num_steps=400)

pred_rel = dominance_relation(q.mu)
precision, recall, f1 = relation_f1(pred_rel, data.relation)

print(f"Objective start: {history[0]:.4f}")
print(f"Objective end  : {history[-1]:.4f}")
print(f"Relation precision={precision:.3f}, recall={recall:.3f}, f1={f1:.3f}")

window = 25
smoothed = np.convolve(history, np.ones(window) / window, mode="valid")
print(f"Smoothed objective (first, last): {smoothed[0]:.4f} -> {smoothed[-1]:.4f}")


Objective start: -3075.2970
Objective end  : -1601.2953
Relation precision=1.000, recall=0.571, f1=0.727
Smoothed objective (first, last): -2147.3334 -> -1724.5710


In [5]:
true_set = set(relation_edges(data.relation))
pred_set = set(relation_edges(pred_rel))

tp = sorted(true_set & pred_set)
fp = sorted(pred_set - true_set)
fn = sorted(true_set - pred_set)

print(f"True edges      : {len(true_set)}")
print(f"Predicted edges : {len(pred_set)}")
print(f"TP/FP/FN        : {len(tp)}/{len(fp)}/{len(fn)}")

print("Sample TP edges:", tp[:8])
print("Sample FP edges:", fp[:8])
print("Sample FN edges:", fn[:8])


True edges      : 28
Predicted edges : 16
TP/FP/FN        : 16/0/12
Sample TP edges: [(0, 1), (0, 6), (0, 7), (1, 7), (2, 4), (2, 6), (2, 7), (3, 4)]
Sample FP edges: []
Sample FN edges: [(0, 2), (0, 3), (0, 4), (0, 5), (1, 2), (1, 3), (1, 4), (1, 5)]


## 4) PO-DSL: transitive reduction and maximal-path factoring

In [6]:
branching_order = [(0, 1), (0, 2), (1, 3), (2, 3)]
stats = order_stats(num_nodes=4, order_edges=branching_order)
paths = build_maximal_paths_from_order(num_nodes=4, order_edges=branching_order)

print("Branching order stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")
print("Maximal paths:", paths)

W_consistent = np.array(
    [
        [0.0, 0.8, 0.3],
        [0.0, 0.0, 0.6],
        [0.0, 0.0, 0.0],
    ]
)
chain_order = [(0, 1), (1, 2)]
chain_paths = build_maximal_paths_from_order(3, chain_order)

score_consistent = h_prime(W_consistent, chain_paths)

W_violate = W_consistent.copy()
W_violate[2, 0] = 0.9  # Introduce a back-edge to violate acyclicity.
score_violate = h_prime(W_violate, chain_paths)

closure = closure_edges(3, chain_order)
naive_violate = path_penalty_naive(W_violate, closure)

print(f"h_prime (consistent W): {score_consistent:.8f}")
print(f"h_prime (violating W) : {score_violate:.8f}")
print(f"naive closure penalty  : {naive_violate:.8f}")


Branching order stats:
  num_order_edges: 4
  num_closure_edges: 5
  num_reduction_edges: 4
  num_maximal_paths: 2
Maximal paths: [[0, 1, 3], [0, 2, 3]]
h_prime (consistent W): 0.00000000
h_prime (violating W) : 0.48355797
naive closure penalty  : 1.67904900


## 5) Scaling table (same idea as `scripts/run_po_dsl_stress.py`)

In [7]:
def chain_edges(length: int) -> list[tuple[int, int]]:
    return [(i, i + 1) for i in range(length - 1)]

print("m\t|O+|\t|O-|\tmax_paths\tnaive_terms\tfactored_terms")
for m in range(10, 121, 20):
    order = chain_edges(m)
    closure = closure_edges(m, order)
    reduction = transitive_reduction(m, order)
    max_paths = build_maximal_paths_from_order(m, order)
    naive_terms = len(closure)
    factored_terms = len(max_paths)
    print(
        f"{m}\t{len(closure)}\t{len(reduction)}\t{len(max_paths)}"
        f"\t{naive_terms}\t{factored_terms}"
    )


m	|O+|	|O-|	max_paths	naive_terms	factored_terms
10	45	9	1	45	1
30	435	29	1	435	1
50	1225	49	1	1225	1
70	2415	69	1	2415	1
90	4005	89	1	4005	1
110	5995	109	1	5995	1


## 6) Summary

This notebook reproduces the full scaffolded workflow: synthetic generation, HPO trace modeling, VI training, relation recovery, and PO-DSL order factoring/scaling.

If you want to explore further, try:
- `structured_order=False` in data generation.
- different `num_particles`, `alpha`, and `lr` in `IWVIConfig`.
- larger chain sizes in the stress-table cell.
